In [1]:
import os, random, shutil

data_dir = "/Users/abdulmateen/Downloads/Dataset/test"
output_dir = "/Users/abdulmateen/Downloads/Dataset/sample_eval"

os.makedirs(output_dir, exist_ok=True)

all_images = []
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(os.path.join(root, f))

# Randomly pick 20
sampled = random.sample(all_images, 20)

# Copy them into a separate folder for evaluation
for img in sampled:
    shutil.copy(img, output_dir)

print("Sampled 20 images to:", output_dir)


Sampled 20 images to: /Users/abdulmateen/Downloads/Dataset/sample_eval


In [ ]:
import requests
import base64
import os
from io import BytesIO
from PIL import Image

# --- Configuration ---
# 1. Replace with your actual Inference Endpoint URL
ENDPOINT_URL = "https://ggmnc9stoewr89x4.us-east-1.aws.endpoints.huggingface.cloud"
# 2. Make sure your Hugging Face token is set as an environment variable
HF_TOKEN = os.getenv("HF_TOKEN")
# ---

def query(payload):
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json"
    }
    # The entire payload is now sent inside the "inputs" key
    response = requests.post(ENDPOINT_URL, headers=headers, json={"inputs": payload})
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"Request failed with status code {response.status_code}", "details": response.text}

# --- Method 1: Using an Image URL ---
print("--- Testing with Image URL ---")
# The payload now contains your data for the handler
payload_url = {
    "prompt": "What are the key features of the lesion in this image?",
    "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6c/Melanoma.jpg/640px-Melanoma.jpg",
    "max_new_tokens": 150,
    "lora_path": "Abdulmateen/llava-finetuned"  
}
output_url = query(payload_url)
print(output_url)

# --- Method 2: Using a Local Image (Base64 Encoded) ---
print("\n--- Testing with Local Image (Base64) ---")
image_path = "/Users/abdulmateen/Downloads/WhatsApp Image 2025-07-31 at 11.38.37.jpeg"

def image_to_base64(path):
    try:
        with open(path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: Image file not found at '{path}'. Skipping Base64 test.")
        return None

base64_string = image_to_base64(image_path)

if base64_string:
    # This payload also contains your data
    payload_b64 = {
        "prompt": "What type of acne is shown in the image?.",
        "image_b64": base64_string,
        "max_new_tokens": 150,
        "lora_path": "Abdulmateen/llava-finetuned"  
    }
    output_b64 = query(payload_b64)
    print(output_b64)